In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

from pathlib import Path
from sklearn.model_selection import train_test_split

sns.set_theme(style="whitegrid")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)

In [2]:
PROJECT_DIR = Path.cwd().parent

RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
SPLIT_DIR = PROJECT_DIR / "data" / "splits"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_DIR)
print("Raw data:", RAW_DIR)

Project: c:\Users\Kowshik\my-project\text-summarization-system
Raw data: c:\Users\Kowshik\my-project\text-summarization-system\data\raw


In [3]:
bbc_path = RAW_DIR / "bbc-news-summary.csv"
cnn_path = RAW_DIR / "cnn_dailymail_test_dataset.csv"

bbc = pd.read_csv(bbc_path)
cnn = pd.read_csv(cnn_path)

print("BBC Shape:", bbc.shape)
print("CNN Shape:", cnn.shape)

BBC Shape: (2224, 3)
CNN Shape: (11490, 3)


In [4]:
def clean_text(text):

    if not isinstance(text, str):
        return ""

    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [5]:
bbc_clean = bbc[["Articles", "Summaries"]].copy()

bbc_clean.columns = ["article", "summary"]

bbc_clean["article"] = bbc_clean["article"].apply(clean_text)
bbc_clean["summary"] = bbc_clean["summary"].apply(clean_text)

# Remove empty records
bbc_clean = bbc_clean[
    (bbc_clean["article"] != "") &
    (bbc_clean["summary"] != "")
]

# Remove duplicate article-summary pairs
bbc_clean = bbc_clean.drop_duplicates(
    subset=["article", "summary"]
)

# Remove repeated articles
bbc_clean = bbc_clean.drop_duplicates(
    subset=["article"]
)

bbc_clean = bbc_clean.reset_index(drop=True)

print("Original BBC:", len(bbc))
print("Clean BBC:", len(bbc_clean))

Original BBC: 2224
Clean BBC: 2126


In [6]:
cnn_clean = cnn[["article", "highlights"]].copy()

cnn_clean.columns = ["article", "summary"]

cnn_clean["article"] = cnn_clean["article"].apply(clean_text)
cnn_clean["summary"] = cnn_clean["summary"].apply(clean_text)

cnn_clean = cnn_clean[
    (cnn_clean["article"] != "") &
    (cnn_clean["summary"] != "")
]

cnn_clean = cnn_clean.drop_duplicates(
    subset=["article", "summary"]
)

cnn_clean = cnn_clean.drop_duplicates(
    subset=["article"]
)

cnn_clean = cnn_clean.reset_index(drop=True)

print("Original CNN:", len(cnn))
print("Clean CNN:", len(cnn_clean))

Original CNN: 11490
Clean CNN: 11488


In [7]:
bbc_model_data = bbc_clean[["article", "summary"]].copy()

In [8]:
train_data, temp_data = train_test_split(
    bbc_model_data,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

validation_data, test_data = train_test_split(
    temp_data,
    test_size=0.50,
    random_state=42,
    shuffle=True
)

In [9]:
print("Training:", len(train_data))
print("Validation:", len(validation_data))
print("Testing:", len(test_data))

print("\nTotal:", len(train_data) + len(validation_data) + len(test_data))

Training: 1700
Validation: 213
Testing: 213

Total: 2126


In [10]:
train_articles = set(train_data["article"])
validation_articles = set(validation_data["article"])
test_articles = set(test_data["article"])

print("Train-Validation Overlap:",
      len(train_articles & validation_articles))

print("Train-Test Overlap:",
      len(train_articles & test_articles))

print("Validation-Test Overlap:",
      len(validation_articles & test_articles))

Train-Validation Overlap: 0
Train-Test Overlap: 0
Validation-Test Overlap: 0


In [11]:
train_data.to_csv(
    SPLIT_DIR / "train.csv",
    index=False
)

validation_data.to_csv(
    SPLIT_DIR / "validation.csv",
    index=False
)

test_data.to_csv(
    SPLIT_DIR / "test.csv",
    index=False
)

cnn_clean[["article", "summary"]].to_csv(
    SPLIT_DIR / "cnn_external_test.csv",
    index=False
)

bbc_clean[["article", "summary"]].to_csv(
    PROCESSED_DIR / "bbc_clean.csv",
    index=False
)

cnn_clean[["article", "summary"]].to_csv(
    PROCESSED_DIR / "cnn_clean.csv",
    index=False
)

print("All datasets saved successfully!")

All datasets saved successfully!


In [12]:
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path.cwd().parent

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

bbc = pd.read_csv(PROCESSED_DIR / "bbc_clean.csv")
cnn = pd.read_csv(PROCESSED_DIR / "cnn_clean.csv")

print("BBC:", bbc.shape)
print("CNN:", cnn.shape)

display(bbc.head(3))
display(cnn.head(3))

BBC: (2126, 2)
CNN: (11488, 2)


,article,summary
0,Budget to set scene for election..Gordon Brown will seek to put the economy at the centre of Labour's bid for a third term in power when he delivers his ninth Budget at 1230 GMT. He is expected to...,"- Increase in the stamp duty threshold from £60,000 - A freeze on petrol duty - An extension of tax credit scheme for poorer families - Possible help for pensioners The stamp duty threshold rise i..."
1,"Army chiefs in regiments decision..Military chiefs are expected to meet to make a final decision on the future of Scotland's Army regiments...A committee of the Army Board, which is made up of the...","""They are very much not for the good and will destroy Scotland's regiments by moulding them into a single super regiment which will lead to severe recruitment problems, a loss of local connections..."
2,Howard denies split over ID cards..Michael Howard has denied his shadow cabinet was split over its decision to back controversial Labour plans to introduce ID cards...The Tory leader said his fron...,"Michael Howard has denied his shadow cabinet was split over its decision to back controversial Labour plans to introduce ID cards.Mr Howard said the police had said ID cards could ""help them foil ..."


,article,summary
0,"(CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian terr...","Membership gives the ICC jurisdiction over alleged crimes committed in Palestinian territories since last June . Israel and the United States opposed the move, which could open the door to war cri..."
1,"(CNN)Never mind cats having nine lives. A stray pooch in Washington State has used up at least three of her own after being hit by a car, apparently whacked on the head with a hammer in a misguide...","Theia, a bully breed mix, was apparently hit by a car, whacked with a hammer and buried in a field . ""She's a true miracle dog and she deserves a good life,"" says Sara Mellado, who is looking for ..."
2,"(CNN)If you've been following the news lately, there are certain things you doubtless know about Mohammad Javad Zarif. He is, of course, the Iranian foreign minister. He has been U.S. Secretary of...",Mohammad Javad Zarif has spent more time with John Kerry than any other foreign minister . He once participated in a takeover of the Iranian Consulate in San Francisco . The Iranian foreign minist...


In [13]:
for name, dataset in [("BBC", bbc), ("CNN", cnn)]:

    print(f"\n{name} Dataset")
    print("Columns:", dataset.columns.tolist())
    print("Missing values:\n", dataset.isnull().sum())
    print("Duplicate articles:", dataset["article"].duplicated().sum())


BBC Dataset
Columns: ['article', 'summary']
Missing values:
 article    0
summary    0
dtype: int64
Duplicate articles: 0

CNN Dataset
Columns: ['article', 'summary']
Missing values:
 article    0
summary    0
dtype: int64
Duplicate articles: 0


In [14]:
from transformers import AutoTokenizer

MODEL_NAME = "t5-small"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded:", MODEL_NAME)

c:\Users\Kowshik\my-project\text-summarization-system\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tokenizer loaded: t5-small


In [15]:
def get_token_length(text):

    tokens = tokenizer(
        str(text),
        truncation=False
    )

    return len(tokens["input_ids"])

In [16]:
bbc["article_tokens"] = bbc["article"].apply(get_token_length)
bbc["summary_tokens"] = bbc["summary"].apply(get_token_length)

cnn["article_tokens"] = cnn["article"].apply(get_token_length)
cnn["summary_tokens"] = cnn["summary"].apply(get_token_length)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (708 > 512). Running this sequence through the model will result in indexing errors


In [17]:
print("BBC Article Token Statistics:")
display(bbc["article_tokens"].describe(
    percentiles=[0.50, 0.90, 0.95, 0.99]
))

print("BBC Summary Token Statistics:")
display(bbc["summary_tokens"].describe(
    percentiles=[0.50, 0.90, 0.95, 0.99]
))

BBC Article Token Statistics:


count    2126.000000
mean      533.279398
std       327.498191
min       117.000000
50%       459.000000
90%       846.500000
95%      1013.750000
99%      1391.000000
max      6002.000000
Name: article_tokens, dtype: float64

BBC Summary Token Statistics:


count    2126.000000
mean      238.852305
std       155.181900
min        52.000000
50%       205.000000
90%       388.000000
95%       450.000000
99%       627.250000
max      2925.000000
Name: summary_tokens, dtype: float64

In [18]:
print("CNN Article Token Statistics:")
display(cnn["article_tokens"].describe(
    percentiles=[0.50, 0.90, 0.95, 0.99]
))

print("CNN Summary Token Statistics:")
display(cnn["summary_tokens"].describe(
    percentiles=[0.50, 0.90, 0.95, 0.99]
))

CNN Article Token Statistics:


count    11488.000000
mean       969.201950
std        497.842275
min         73.000000
50%        870.000000
90%       1689.000000
95%       1973.650000
99%       2436.000000
max       3688.000000
Name: article_tokens, dtype: float64

CNN Summary Token Statistics:


count    11488.000000
mean        79.300313
std         32.373987
min         12.000000
50%         74.000000
90%        116.000000
95%        135.000000
99%        174.000000
max        968.000000
Name: summary_tokens, dtype: float64

In [19]:
from transformers import AutoTokenizer

MODEL_NAME = "t5-small"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded successfully!")

Tokenizer loaded successfully!


In [20]:
sample_text = "Artificial intelligence is changing the world."

tokens = tokenizer(sample_text)

print("Original text:", sample_text)
print("Token IDs:", tokens["input_ids"])
print("Token count:", len(tokens["input_ids"]))

Original text: Artificial intelligence is changing the world.
Token IDs: [24714, 6123, 19, 2839, 8, 296, 5, 1]
Token count: 8


In [21]:
def get_article_token_length(text):
    text = "summarize: " + str(text)
    return len(tokenizer(text, truncation=False)["input_ids"])


def get_summary_token_length(text):
    return len(tokenizer(str(text), truncation=False)["input_ids"])

In [22]:
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\Kowshik\my-project\text-summarization-system")
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

print("Processed folder exists:", PROCESSED_DIR.exists())

print("\nAvailable CSV files:")
for file in PROCESSED_DIR.glob("*.csv"):
    print(file.name)

Processed folder exists: True

Available CSV files:
bbc_clean.csv
cnn_clean.csv


In [23]:
bbc["article_tokens"] = bbc["article"].apply(get_article_token_length)
bbc["summary_tokens"] = bbc["summary"].apply(get_summary_token_length)

cnn["article_tokens"] = cnn["article"].apply(get_article_token_length)
cnn["summary_tokens"] = cnn["summary"].apply(get_summary_token_length)

print("Token length calculation completed.")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (710 > 512). Running this sequence through the model will result in indexing errors


Token length calculation completed.


In [24]:
percentiles = [0.50, 0.90, 0.95, 0.99]

print("BBC Article Token Lengths")
display(bbc["article_tokens"].describe(percentiles=percentiles))

print("BBC Summary Token Lengths")
display(bbc["summary_tokens"].describe(percentiles=percentiles))

print("CNN Article Token Lengths")
display(cnn["article_tokens"].describe(percentiles=percentiles))

print("CNN Summary Token Lengths")
display(cnn["summary_tokens"].describe(percentiles=percentiles))

BBC Article Token Lengths


count    2126.000000
mean      535.279398
std       327.498191
min       119.000000
50%       461.000000
90%       848.500000
95%      1015.750000
99%      1393.000000
max      6004.000000
Name: article_tokens, dtype: float64

BBC Summary Token Lengths


count    2126.000000
mean      238.852305
std       155.181900
min        52.000000
50%       205.000000
90%       388.000000
95%       450.000000
99%       627.250000
max      2925.000000
Name: summary_tokens, dtype: float64

CNN Article Token Lengths


count    11488.000000
mean       971.201950
std        497.842275
min         75.000000
50%        872.000000
90%       1691.000000
95%       1975.650000
99%       2438.000000
max       3690.000000
Name: article_tokens, dtype: float64

CNN Summary Token Lengths


count    11488.000000
mean        79.300313
std         32.373987
min         12.000000
50%         74.000000
90%        116.000000
95%        135.000000
99%        174.000000
max        968.000000
Name: summary_tokens, dtype: float64